# Notebook 02 — Colector Interactivo de Puntos de Control y Partición 70/30 (100% Local SEPAL)

- **Instancia SEPAL recomendada:** `t2` o `m2` (muy liviano; todo corre en memoria y disco local en < 1 segundo).
- **Cómo funciona:**
  1. Visualiza el Stack Biestacional local + imagen satelital de alta resolución + capa guía `DinMangle_2020_2022_V23` local.
  2. Opcionalmente genera puntos candidatos de Mangle en el núcleo erosionado (`20 m`) para inspección y complemento manual.
  3. Dibuja directamente en el mapa puntos de las **5 clases del Protocolo** (`1: Mangle`, `2: Latifoliado`, `3: Agua`, `4: Urbano`, `5: Pastos`), deduplicando automáticamente en celdas exactas de 10 m (`EPSG:32615` / `EPSG:32616`).
  4. Realiza el split preliminar **70/30** (solo diagnóstico rápido) y extrae al instante las **24 bandas** directamente desde tu GeoTIFF local (`outputs/stack/...tif`), sin esperar a GEE.

In [ ]:
# 1. Configuracion de region y carga de Stack + Guia locales
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import geemap
import geopandas as gpd
import ipywidgets as widgets
import numpy as np
import pandas as pd
from IPython.display import clear_output, display
from shapely.geometry import shape
from sklearn.model_selection import train_test_split

from src.config_utils import ensure_output_dirs, get_class_maps, load_config
from src.local_aoi import ensure_local_aoi, get_local_guia_path
from src.local_rf import extract_bands_at_points, propose_mangle_candidates_local
from src.local_stack import add_local_raster_overlay, get_local_stack_paths

# >>> CAMBIAR AQUI ENTRE "PACIFICO" Y "CARIBE" <<<
REGION = "PACIFICO"

cfg = load_config()
dirs = ensure_output_dirs(cfg)
CLASES_NOMBRES, CLASES_CORTOS, CLASES_COLORES = get_class_maps(cfg)

paths = get_local_stack_paths(REGION, cfg)
STACK_PATH = paths["stack"]
QA_PATH = paths["qa"]
GUIA_PATH = get_local_guia_path(REGION, cfg)
aoi_gdf = ensure_local_aoi(REGION, cfg)

assert STACK_PATH.is_file(), f"No se encontro el Stack local {STACK_PATH}. Ejecuta primero el Notebook 01."
GPKG_RAW = dirs["muestras"] / f"muestras_control_raw_{REGION.lower()}.gpkg"
print(f"Stack local listo : {STACK_PATH.name}")
print(f"Capa guia local   : {GUIA_PATH.name if GUIA_PATH.is_file() else 'No encontrada (puedes dibujar sin ella)'}")
print(f"Archivo de puntos : {GPKG_RAW}")

In [ ]:
# 2. (Opcional) Proponer puntos candidatos dentro de DinMangle_2020_2022_V23 erosionado 20 m
GENERAR_CANDIDATOS_MANGLE = False
N_CANDIDATOS_MANGLE = 120
VALOR_CLASE_MANGLE_EN_GUIA = 1  # Ajustar si en DinMangle la clase manglar tiene otro codigo entero

def cargar_puntos_existentes(path_gpkg: Path) -> gpd.GeoDataFrame:
    if path_gpkg.exists():
        gdf = gpd.read_file(path_gpkg)
        if gdf.crs is None:
            gdf = gdf.set_crs("EPSG:4326")
        return gdf.to_crs("EPSG:4326")
    return gpd.GeoDataFrame(
        columns=["clase_id", "clase", "fuente", "region", "geometry"],
        geometry="geometry",
        crs="EPSG:4326",
    )

gdf_puntos = cargar_puntos_existentes(GPKG_RAW)

if GENERAR_CANDIDATOS_MANGLE:
    assert GUIA_PATH.is_file(), f"Descarga primero DinMangle en el Notebook 00: {GUIA_PATH}"
    nuevos = propose_mangle_candidates_local(
        GUIA_PATH,
        QA_PATH,
        region=REGION,
        n_points=N_CANDIDATOS_MANGLE,
        mangle_val=VALOR_CLASE_MANGLE_EN_GUIA,
        erosion_m=float(cfg["muestreo"]["erosion_mangle_guia_m"]),
        seed=int(cfg["proyecto"]["seed"]),
    )
    gdf_puntos = gpd.GeoDataFrame(pd.concat([gdf_puntos, nuevos], ignore_index=True), geometry="geometry", crs="EPSG:4326")
    gdf_puntos.to_file(GPKG_RAW, driver="GPKG")
    print(f"[OK] Se agregaron {len(nuevos)} candidatos de Mangle en {GPKG_RAW.name} (<1s local).")
else:
    print(f"Puntos actuales guardados en {GPKG_RAW.name}: {len(gdf_puntos)}")
    if len(gdf_puntos) > 0:
        display(gdf_puntos.groupby(["clase_id", "clase"]).size().rename("n_puntos").to_frame())

In [ ]:
# 3. COLECTOR INTERACTIVO DE PUNTOS DE CONTROL EN JUPYTER (100% local en SEPAL)
m = geemap.Map(height="620px")
m.add_basemap("SATELLITE")
b = aoi_gdf.total_bounds
m.setCenter((b[0] + b[2]) / 2.0, (b[1] + b[3]) / 2.0, 11)

add_local_raster_overlay(m, STACK_PATH, ["B8_SECA", "B11_SECA", "B4_SECA"], "Falsocolor Manglar (B8-B11-B4 Seca)", stretch=(0.0, 0.40), visible=True)
add_local_raster_overlay(m, STACK_PATH, ["B4_SECA", "B3_SECA", "B2_SECA"], "S2 Seca RGB", stretch=(0.0, 0.25), visible=False)
add_local_raster_overlay(m, STACK_PATH, ["VH_SECA"], "SAR VH Seca (dB)", stretch=(-28.0, -8.0), palette=["#0015FF", "#00FFFF", "#FFFF00", "#FF0000"], visible=False)
if GUIA_PATH.is_file():
    add_local_raster_overlay(m, GUIA_PATH, [1], "Guia DinMangle 2020-2022", categorical=True, palette={1: "#00FF66"}, opacity=0.55, visible=True)
m.add_gdf(aoi_gdf, layer_name=f"AOI {REGION}", style={"color": "#FFFFFF", "fillOpacity": 0.0, "weight": 2})

def _dibujar_puntos_guardados():
    gdf_curr = cargar_puntos_existentes(GPKG_RAW)
    if len(gdf_curr) == 0:
        return
    for cid, grp in gdf_curr.groupby("clase_id"):
        cid_int = int(cid)
        m.add_gdf(grp[["clase_id", "clase", "geometry"]], layer_name=f"Puntos {cid_int}: {CLASES_NOMBRES[cid_int]} ({len(grp)})", style={"color": CLASES_COLORES[cid_int], "fillColor": CLASES_COLORES[cid_int]})

_dibujar_puntos_guardados()

dd_clase = widgets.Dropdown(
    options=[(f"{cid} - {nom}", cid) for cid, nom in CLASES_NOMBRES.items()],
    value=1,
    description="Clase:",
    style={"description_width": "initial"},
)
btn_guardar = widgets.Button(description="Guardar puntos dibujados", button_style="success", icon="save")
btn_deshacer = widgets.Button(description="Deshacer ultimo lote", button_style="warning", icon="undo")
out_log = widgets.Output()
HISTORIAL_LOTES = []

def _mostrar_resumen(gdf):
    if len(gdf) == 0:
        print("Aun no hay puntos guardados.")
        return
    display(gdf.groupby(["clase_id", "clase"]).size().rename("n_puntos").reset_index())
    print(f"Total acumulado: {len(gdf)} puntos ({GPKG_RAW.name})")

def on_guardar_click(_):
    with out_log:
        clear_output(wait=True)
        features = list(getattr(m.draw_control, "data", None) or [])
        if not features:
            features = list(getattr(m, "draw_features", None) or [])
        if not features:
            print("No hay geometrias dibujadas en el mapa. Dibuja al menos un marcador o poligono primero.")
            return
        cid = int(dd_clase.value)
        nuevos_rows = []
        for feat in features:
            geom_obj = shape(feat["geometry"]) if isinstance(feat, dict) else shape(feat.geometry().getInfo())
            if geom_obj.geom_type == "Point":
                pt = geom_obj
            elif geom_obj.geom_type == "MultiPoint":
                for p in geom_obj.geoms:
                    nuevos_rows.append({"clase_id": cid, "clase": CLASES_NOMBRES[cid], "fuente": "dibujo_manual_sepal", "region": REGION, "geometry": p})
                continue
            else:
                pt = geom_obj.representative_point()
            nuevos_rows.append({"clase_id": cid, "clase": CLASES_NOMBRES[cid], "fuente": "dibujo_manual_sepal", "region": REGION, "geometry": pt})

        gdf_curr = cargar_puntos_existentes(GPKG_RAW)
        gdf_new = gpd.GeoDataFrame(nuevos_rows, geometry="geometry", crs="EPSG:4326")
        gdf_all = gpd.GeoDataFrame(pd.concat([gdf_curr, gdf_new], ignore_index=True), geometry="geometry", crs="EPSG:4326")
        gdf_all.to_file(GPKG_RAW, driver="GPKG")
        HISTORIAL_LOTES.append(len(gdf_new))

        m.draw_control.clear()
        if hasattr(m, "draw_features"):
            m.draw_features = []
        if hasattr(m, "user_rois"):
            m.user_rois = None
        try:
            m.add_gdf(gdf_new[["clase_id", "clase", "geometry"]], layer_name=f"Lote {len(HISTORIAL_LOTES)}: {CLASES_NOMBRES[cid]} ({len(gdf_new)})")
        except Exception:
            pass
        print(f"[OK] Guardados {len(gdf_new)} punto(s) en Clase {cid} ({CLASES_NOMBRES[cid]}).")
        _mostrar_resumen(gdf_all)

def on_deshacer_click(_):
    with out_log:
        clear_output(wait=True)
        if not HISTORIAL_LOTES:
            print("No hay lotes recientes en esta sesion para deshacer.")
            return
        n_rem = HISTORIAL_LOTES.pop()
        gdf_curr = cargar_puntos_existentes(GPKG_RAW).iloc[:-n_rem].copy()
        gdf_curr.to_file(GPKG_RAW, driver="GPKG")
        print(f"Se eliminaron los ultimos {n_rem} punto(s).")
        _mostrar_resumen(gdf_curr)

btn_guardar.on_click(on_guardar_click)
btn_deshacer.on_click(on_deshacer_click)

display(widgets.HBox([dd_clase, btn_guardar, btn_deshacer]))
display(out_log)
display(m)

In [ ]:
# 4. Deduplicacion espacial (celda 10m), Split 70/30 y extraccion instantanea de las 24 bandas
gdf_raw = cargar_puntos_existentes(GPKG_RAW)
assert len(gdf_raw) > 0, "Primero colecta puntos de control en la celda anterior."

crs_utm = cfg["regiones"][REGION]["crs"]
gdf_utm = gdf_raw.to_crs(crs_utm).copy()
gdf_utm["px_col"] = np.floor(gdf_utm.geometry.x / 10.0).astype(int)
gdf_utm["px_row"] = np.floor(gdf_utm.geometry.y / 10.0).astype(int)
gdf_clean = gdf_utm.drop_duplicates(subset=["px_col", "px_row"], keep="first").copy().to_crs("EPSG:4326")

# Extraer al instante las 24 bandas desde el Stack GeoTIFF local (solo pixeles validos)
gdf_sampled = extract_bands_at_points(STACK_PATH, gdf_clean, qa_path=QA_PATH, require_base_aoi=False)
gdf_sampled["lon"] = gdf_sampled.geometry.x
gdf_sampled["lat"] = gdf_sampled.geometry.y
gdf_sampled["sample_id"] = np.arange(1, len(gdf_sampled) + 1)

conteos = gdf_sampled["clase_id"].value_counts().sort_index()
print(f"Brutos: {len(gdf_raw)} -> Deduplicados y validos en Stack: {len(gdf_sampled)}")
display(conteos.rename("n_validos").to_frame())

assert (conteos >= 4).all() and len(conteos) >= 2, "Asegurate de tener al menos 4 puntos validos en cada clase antes de continuar."

idx_train, idx_test = train_test_split(
    gdf_sampled.index,
    test_size=float(cfg["muestreo"]["test_size_preliminar"]),
    stratify=gdf_sampled["clase_id"],
    random_state=int(cfg["proyecto"]["seed"]),
)
gdf_sampled["split"] = "train"
gdf_sampled.loc[idx_test, "split"] = "test_preliminar"

display(pd.crosstab(gdf_sampled["clase"], gdf_sampled["split"], margins=True))

out_csv = dirs["muestras"] / f"muestras_limpias_{REGION.lower()}_70_30_con_bandas.csv"
out_gpkg = dirs["muestras"] / f"muestras_limpias_{REGION.lower()}_70_30.gpkg"
gdf_sampled.drop(columns="geometry").to_csv(out_csv, index=False)
gdf_sampled[["sample_id", "clase_id", "clase", "split", "fuente", "region", "lon", "lat", "geometry"]].to_file(out_gpkg, driver="GPKG")

print(f"[OK] Tabla lista con las 24 bandas (<1s, sin GEE): {out_csv}")
print(f"[OK] GeoPackage EPSG:4326 guardado en            : {out_gpkg}")